# Wikimedia Silver Layer Pipeline

### Purpose
This notebook implements the Silver layer of a Wikimedia data pipeline using a medallion (Bronze → Silver) architecture.\
It reads raw Wikimedia edit events from a Bronze Delta table, deduplicates them, and upserts them into a Silver Delta table.

### Prerequisites
- Creates the Silver Delta table with columns: `id`, `title`, `comment`, `title_url`, `timestamp`, `user`, `wiki`.
- Enables Change Data Feed (CDF) on the Bronze table so row-level changes are captured for downstream consumption.

### Core streaming upsert logic 
- defines merge_to_silver() which deduplicates micro-batches by `(wiki, title)` keeping the latest event (by `timestamp`), then MERGEs into Silver — updating matched rows only when `t.timestamp < s.timestamp`, and inserting new rows otherwise.
- A readStream reads from Bronze, casts `timestamp` to timestamp type, filters out bots (`bot == false`) and keeps only `edit`/`new` event types, drops rows with null `id`, projects the target columns, and writes via `foreachBatch` with `trigger(availableNow=True)`.


In [0]:
%run ./params_config

### Create table

spark.sql(f"CREATE OR REPLACE TABLE {catalog}.{silver_schema}.{silver_table} ( id long, title string,  comment string, title_url string, timestamp  timestamp , user string,  wiki string)")


###  Enable cdf to produce feed table

```sql
ALTER TABLE {catalog}.{bronze_schema}.{bronze_table}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

This enables Change Data Feed on the Bronze table so that row-level changes (inserts, updates, deletes) are captured and can be consumed downstream by the Silver layer pipeline.

In [0]:
def merge_to_silver(microBatchDF, batch_id):
    # dedupe within the microbatch: keep latest event per page key
    w = Window.partitionBy("wiki", "title").orderBy(col("timestamp").desc())

    deduped = (microBatchDF
               .withColumn("rn", row_number().over(w))
               .filter("rn = 1")
               .drop("rn"))

    deduped.createOrReplaceTempView("wikimedia")

    spark.sql(f"""
        MERGE INTO {catalog}.{silver_schema}.{silver_table} t
        USING wikimedia s
        ON t.wiki = s.wiki AND t.title = s.title

        WHEN MATCHED AND t.timestamp < s.timestamp
          THEN UPDATE SET *

        WHEN NOT MATCHED
          THEN INSERT *
    """)

# trigger(availableNow=True) processes all data available at start time, then the
# query terminates. Combined with awaitTermination(), the cell exits and the stream
# stops. This is expected behaviour — it is a one-shot / scheduled-batch pattern.
#
# To keep the stream running continuously, switch to a processingTime trigger
# (e.g. .trigger(processingTime="10s")) and drop awaitTermination() so the query
# runs in the background.

(spark.readStream
    .table(f"{catalog}.{bronze_schema}.{bronze_table}")
    .filter((col("bot") == False) & (col("type").isin("edit", "new")))
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
    .filter(col("id").isNotNull())
    .select("id", "title", "comment", "title_url", "timestamp", "type", "user", "wiki")
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(merge_to_silver)
    .option("checkpointLocation", f"{checkpoint_silver}")
    .start()
    .awaitTermination())